In [1]:
# ============================================================================
# PROJET : Detection des Mauvaises Herbes par Deep Learning
# VERSION KAGGLE
# ============================================================================
#
# INSTRUCTIONS KAGGLE :
# 1. Ouvrez un nouveau Notebook sur https://www.kaggle.com/
# 2. Dans l'onglet "Data" (a droite) > "Add Data"
#    Recherchez "v2-plant-seedlings-dataset" et ajoutez-le
# 3. Activez le GPU : Settings > Accelerator > GPU T4 x2 (ou P100)
# 4. Copiez-collez ce code dans les cellules ou importez ce fichier
# ============================================================================

In [ ]:
!python -m pip install --upgrade pip
!pip install matplotlib seaborn pandas scikit-learn Pillow
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.9 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
# ============================================================================
# 1. IMPORTATION DES BIBLIOTHEQUES
# ============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
import random
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("=" * 60)
print("  DETECTION DES MAUVAISES HERBES - VISION PAR ORDINATEUR")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

2026-02-23 20:58:30.618133: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771880310.813387      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771880310.868030      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771880311.308468      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771880311.308514      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771880311.308517      55 computation_placer.cc:177] computation placer alr

  DETECTION DES MAUVAISES HERBES - VISION PAR ORDINATEUR
TensorFlow version: 2.19.0
GPU disponible: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [4]:
# ============================================================================
# 2. CONFIGURATION
# ============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [5]:
DATASET_PATH = "/kaggle/input/datasets/vbookshelf/v2-plant-seedlings-dataset"
print(f"[OK] Dataset : {DATASET_PATH}")

CLASS_NAMES = [
    'Black-grass', 'Charlock', 'Cleavers', 'Common Chickweed',
    'Common wheat', 'Fat Hen', 'Loose Silky-bent', 'Maize',
    'Scentless Mayweed', 'Shepherds Purse',
    'Small-flowered Cranesbill', 'Sugar beet'
]
NUM_CLASSES = len(CLASS_NAMES)
WEEDS = ['Black-grass', 'Charlock', 'Cleavers', 'Common Chickweed',
         'Fat Hen', 'Loose Silky-bent', 'Scentless Mayweed',
         'Shepherds Purse', 'Small-flowered Cranesbill']
CROPS = ['Common wheat', 'Maize', 'Sugar beet']

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

[OK] Dataset : /kaggle/input/datasets/vbookshelf/v2-plant-seedlings-dataset


In [6]:
# ============================================================================
# 3. CHARGEMENT DES IMAGES
# ============================================================================
def load_dataset(path):
    images, labels, samples_per_class = [], [], {}
    for cls_name in CLASS_NAMES:
        cls_path = os.path.join(path, cls_name)
        if not os.path.exists(cls_path):
            print(f"  [!] Classe manquante : {cls_name}")
            continue
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        samples_per_class[cls_name] = len(files)
        for f in files:
            try:
                img = Image.open(os.path.join(cls_path, f)).convert('RGB')
                img = img.resize((IMG_SIZE, IMG_SIZE))
                images.append(np.array(img, dtype=np.float32))
                labels.append(CLASS_NAMES.index(cls_name))
            except Exception as e:
                print(f"Erreur chargement {f}: {e}")
    return np.array(images, dtype=np.float32), np.array(labels), samples_per_class

print("Chargement des images...")
X_data, y_data, samples_per_class = load_dataset(DATASET_PATH)
print(f"[OK] {len(X_data)} images chargees")

# Preprocessing MobileNetV2 : [0, 255] -> [-1, 1]
X_data = preprocess_input(X_data)
print(f"[OK] Preprocessing applique (plage: [{X_data.min():.1f}, {X_data.max():.1f}])")

Chargement des images...
  [!] Classe manquante : Shepherds Purse
[OK] 5265 images chargees
[OK] Preprocessing applique (plage: [-1.0, 1.0])


In [7]:
# ============================================================================
# 4. SPLIT TRAIN / VAL / TEST (70% / 15% / 15%)
# ============================================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.3, random_state=SEED, stratify=y_data
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# One-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

# Class weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(zip(np.unique(y_train), class_weights_array))
for i in range(NUM_CLASSES):
    if i not in class_weight_dict:
        class_weight_dict[i] = 1.0

print("[OK] Class weights calcules :")
for i, name in enumerate(CLASS_NAMES):
    print(f"     {name}: {class_weight_dict.get(i, 0):.3f}")

Train: 3685, Val: 790, Test: 790
[OK] Class weights calcules :
     Black-grass: 1.551
     Charlock: 1.060
     Cleavers: 1.426
     Common Chickweed: 0.671
     Common wheat: 1.893
     Fat Hen: 0.889
     Loose Silky-bent: 0.629
     Maize: 1.861
     Scentless Mayweed: 0.788
     Shepherds Purse: 1.000
     Small-flowered Cranesbill: 0.831
     Sugar beet: 1.034


In [8]:
# ============================================================================
# 5. DATA AUGMENTATION
# ============================================================================
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator()

In [9]:
# ============================================================================
# 6. CONSTRUCTION DU MODELE (MobileNetV2 + Transfer Learning)
# ============================================================================
base_model = MobileNetV2(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu',
                 kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

I0000 00:00:1771880573.314336      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771880573.320367      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,625,484 (10.02 MB)

 Trainable params: 364,940 (1.39 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [10]:
# ============================================================================
# 7. CALLBACKS
# ============================================================================
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=8,
    restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=4, min_lr=1e-7, verbose=1
)
# Sauvegarde du meilleur modele automatiquement pendant l'entrainement
checkpoint = callbacks.ModelCheckpoint(
    filepath=os.path.join(OUTPUT_DIR, 'best_model.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

In [11]:
# ============================================================================
# 8. ENTRAINEMENT PHASE 1 (Feature Extraction)
# ============================================================================
print("\n" + "=" * 60)
print("  PHASE 1 : Feature Extraction (base model gele)")
print("=" * 60)

EPOCHS_P1 = 30
history = model.fit(
    train_datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    validation_data=val_datagen.flow(X_val, y_val_cat, batch_size=BATCH_SIZE),
    epochs=EPOCHS_P1,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)


  PHASE 1 : Feature Extraction (base model gele)
Epoch 1/30


I0000 00:00:1771880724.362607     137 service.cc:152] XLA service 0x7cd7f01123d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771880724.362645     137 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1771880724.362649     137 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1771880725.540365     137 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-02-23 21:05:33.615979: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-02-23 21:05:33.753273: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1771880736.657733     137 device_co

 78/116 ━━━━━━━━━━━━━━━━━━━━ 10s 289ms/step - accuracy: 0.1028 - loss: 9.3768

2026-02-23 21:06:06.833461: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-02-23 21:06:06.983997: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.1162 - loss: 9.2390

...

116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 303ms/step - accuracy: 0.6870 - loss: 4.1616 - val_accuracy: 0.7975 - val_loss: 3.7782 - learning_rate: 1.0000e-04
Epoch 19/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.6904 - loss: 3.9962
Epoch 19: val_accuracy improved from 0.79747 to 0.80380, saving model to /kaggle/working/best_model.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 303ms/step - accuracy: 0.6904 - loss: 3.9958 - val_accuracy: 0.8038 - val_loss: 3.6406 - learning_rate: 1.0000e-04
Epoch 20/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.6940 - loss: 3.8389
Epoch 20: val_accuracy did not improve from 0.80380
116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 300ms/step - accuracy: 0.6940 - loss: 3.8387 - val_accuracy: 0.7886 - val_loss: 3.5068 - learning_rate: 1.0000e-04
Epoch 21/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.7053 - loss: 3.7137
Epoch 21: val_accuracy improved from 0.80380 to 0.81013, saving model to /kaggle/working/best_model.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 36s 312ms/step - accuracy: 0.7053 - loss: 3.7134 - val_accuracy: 0.8101 - val_loss: 3.3791 - learning_rate: 1.0000e-04
Epoch 22/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step - accuracy: 0.7014 - loss: 3.5945
Epoch 22: val_accuracy improved from 0.81013 to 0.81646, saving model to /kaggle/working/best_model.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 36s 307ms/step - accuracy: 0.7015 - loss: 3.5940 - val_accuracy: 0.8165 - val_loss: 3.2458 - learning_rate: 1.0000e-04
Epoch 23/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.7242 - loss: 3.4474
Epoch 23: val_accuracy did not improve from 0.81646
116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 299ms/step - accuracy: 0.7241 - loss: 3.4472 - val_accuracy: 0.8139 - val_loss: 3.1398 - learning_rate: 1.0000e-04
Epoch 24/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.7240 - loss: 3.3012
Epoch 24: val_accuracy did not improve from 0.81646
116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 299ms/step - accuracy: 0.7240 - loss: 3.3009 - val_accuracy: 0.8139 - val_loss: 3.0231 - learning_rate: 1.0000e-04
Epoch 25/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.7246 - loss: 3.1795
Epoch 25: val_accuracy did not improve from 0.81646
116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 294ms/step - accuracy: 0.7246 - loss: 3.1793 - val_accuracy: 0.8089 - val_loss: 2.9074 - learning_ra

116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 299ms/step - accuracy: 0.7356 - loss: 3.0827 - val_accuracy: 0.8266 - val_loss: 2.7898 - learning_rate: 1.0000e-04
Epoch 27/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.7432 - loss: 2.9384
Epoch 27: val_accuracy did not improve from 0.82658
116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 297ms/step - accuracy: 0.7432 - loss: 2.9383 - val_accuracy: 0.8177 - val_loss: 2.6969 - learning_rate: 1.0000e-04
Epoch 28/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.7611 - loss: 2.8069
Epoch 28: val_accuracy did not improve from 0.82658
116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 297ms/step - accuracy: 0.7611 - loss: 2.8067 - val_accuracy: 0.8139 - val_loss: 2.5924 - learning_rate: 1.0000e-04
Epoch 29/30
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.7508 - loss: 2.7352
Epoch 29: val_accuracy did not improve from 0.82658
116/116 ━━━━━━━━━━━━━━━━━━━━ 36s 307ms/step - accuracy: 0.7508 - loss: 2.7351 - val_accuracy: 0.8127 - val_loss: 2.5055 - learning_ra

In [12]:
# ============================================================================
# 9. ENTRAINEMENT PHASE 2 (Fine-tuning)
# ============================================================================
print("\n" + "=" * 60)
print("  PHASE 2 : Fine-tuning (30 dernieres couches)")
print("=" * 60)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_ft = callbacks.EarlyStopping(
    monitor='val_loss', patience=8,
    restore_best_weights=True, verbose=1
)
reduce_lr_ft = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=4, min_lr=1e-7, verbose=1
)
checkpoint_ft = callbacks.ModelCheckpoint(
    filepath=os.path.join(OUTPUT_DIR, 'best_model_finetuned.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

EPOCHS_P2 = 20
history_ft = model.fit(
    train_datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    validation_data=val_datagen.flow(X_val, y_val_cat, batch_size=BATCH_SIZE),
    epochs=EPOCHS_P2,
    class_weight=class_weight_dict,
    callbacks=[early_stop_ft, reduce_lr_ft, checkpoint_ft],
    verbose=1
)


  PHASE 2 : Fine-tuning (30 dernieres couches)
Epoch 1/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step - accuracy: 0.6137 - loss: 2.9784
Epoch 1: val_accuracy improved from -inf to 0.72658, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 65s 421ms/step - accuracy: 0.6139 - loss: 2.9778 - val_accuracy: 0.7266 - val_loss: 2.7259 - learning_rate: 1.0000e-05
Epoch 2/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.6934 - loss: 2.7748
Epoch 2: val_accuracy did not improve from 0.72658
116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 296ms/step - accuracy: 0.6933 - loss: 2.7746 - val_accuracy: 0.6899 - val_loss: 2.9191 - learning_rate: 1.0000e-05
Epoch 3/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.7050 - loss: 2.6963
Epoch 3: val_accuracy did not improve from 0.72658
116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 299ms/step - accuracy: 0.7050 - loss: 2.6963 - val_accuracy: 0.7038 - val_loss: 2.8838 - learning_rate: 1.0000e-05
Epoch 4/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.7225 - loss: 2.6363
Epoch 4: val_accuracy improved from 0.72658 to 0.72785, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 303ms/step - accuracy: 0.7226 - loss: 2.6362 - val_accuracy: 0.7278 - val_loss: 2.7996 - learning_rate: 1.0000e-05
Epoch 5/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.7331 - loss: 2.5789
Epoch 5: val_accuracy improved from 0.72785 to 0.75190, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 300ms/step - accuracy: 0.7331 - loss: 2.5788 - val_accuracy: 0.7519 - val_loss: 2.6862 - learning_rate: 1.0000e-05
Epoch 6/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.7681 - loss: 2.5371
Epoch 6: val_accuracy improved from 0.75190 to 0.77975, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 305ms/step - accuracy: 0.7681 - loss: 2.5370 - val_accuracy: 0.7797 - val_loss: 2.5553 - learning_rate: 1.0000e-05
Epoch 7/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.7627 - loss: 2.5126
Epoch 7: val_accuracy improved from 0.77975 to 0.79241, saving model to /kaggle/working/best_model_finetuned.h5


...

116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 305ms/step - accuracy: 0.8179 - loss: 2.2866 - val_accuracy: 0.8570 - val_loss: 2.1683 - learning_rate: 1.0000e-05
Epoch 16/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.8179 - loss: 2.2609
Epoch 16: val_accuracy improved from 0.85696 to 0.85823, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 302ms/step - accuracy: 0.8180 - loss: 2.2608 - val_accuracy: 0.8582 - val_loss: 2.1435 - learning_rate: 1.0000e-05
Epoch 17/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.8101 - loss: 2.2601
Epoch 17: val_accuracy did not improve from 0.85823
116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 297ms/step - accuracy: 0.8102 - loss: 2.2599 - val_accuracy: 0.8582 - val_loss: 2.1218 - learning_rate: 1.0000e-05
Epoch 18/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - accuracy: 0.8264 - loss: 2.2108
Epoch 18: val_accuracy improved from 0.85823 to 0.86835, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 34s 297ms/step - accuracy: 0.8264 - loss: 2.2108 - val_accuracy: 0.8684 - val_loss: 2.1001 - learning_rate: 1.0000e-05
Epoch 19/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - accuracy: 0.8293 - loss: 2.2003
Epoch 19: val_accuracy improved from 0.86835 to 0.87595, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 299ms/step - accuracy: 0.8292 - loss: 2.2003 - val_accuracy: 0.8759 - val_loss: 2.0767 - learning_rate: 1.0000e-05
Epoch 20/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - accuracy: 0.8298 - loss: 2.1836
Epoch 20: val_accuracy improved from 0.87595 to 0.88228, saving model to /kaggle/working/best_model_finetuned.h5


116/116 ━━━━━━━━━━━━━━━━━━━━ 35s 298ms/step - accuracy: 0.8299 - loss: 2.1836 - val_accuracy: 0.8823 - val_loss: 2.0509 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 20.


In [13]:
# ============================================================================
# 10. SAUVEGARDE DU MODELE FINAL
# ============================================================================
model_path = os.path.join(OUTPUT_DIR, 'weed_detection_model.h5')
model.save(model_path)
print(f"[OK] Modele sauvegarde : {model_path}")

[OK] Modele sauvegarde : /kaggle/working/weed_detection_model.h5


In [14]:
# ============================================================================
# 11. EVALUATION
# ============================================================================
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
y_pred = np.argmax(model.predict(X_test), axis=1)

all_labels = list(range(NUM_CLASSES))
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                                labels=all_labels, output_dict=True)
cm = confusion_matrix(y_test, y_pred, labels=all_labels)

print(f"\nTest Accuracy: {test_acc*100:.2f}%, Loss: {test_loss:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, labels=all_labels))

25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 195ms/step

Test Accuracy: 88.86%, Loss: 2.0278
                           precision    recall  f1-score   support

              Black-grass       0.61      0.61      0.61        46
                 Charlock       0.97      0.96      0.96        68
                 Cleavers       0.96      0.96      0.96        50
         Common Chickweed       0.95      0.89      0.92       107
             Common wheat       0.87      0.89      0.88        38
                  Fat Hen       0.95      0.79      0.86        80
         Loose Silky-bent       0.81      0.83      0.82       115
                    Maize       0.95      0.95      0.95        39
        Scentless Mayweed       0.82      0.96      0.88        91
          Shepherds Purse       0.00      0.00      0.00         0
Small-flowered Cranesbill       0.94      0.95      0.95        87
               Sugar beet       0.93      0.96      0.94        69

                 accuracy                        

In [15]:
# ============================================================================
# 12. VISUALISATIONS (sauvegardees dans /kaggle/working/)
# ============================================================================

def save_fig(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[OK] {filename}")

# --- 12.1 Distribution des classes ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
counts = [samples_per_class.get(c, 0) for c in CLASS_NAMES]
colors = ['#e74c3c' if c in WEEDS else '#27ae60' for c in CLASS_NAMES]
axes[0].barh(CLASS_NAMES, counts, color=colors)
axes[0].set_xlabel("Nombre d'images")
axes[0].set_title('Distribution des classes')
axes[0].legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='#e74c3c', label='Mauvaise herbe'),
    plt.Rectangle((0, 0), 1, 1, color='#27ae60', label='Culture')
], loc='lower right')
weed_count = sum(samples_per_class.get(w, 0) for w in WEEDS)
crop_count = sum(samples_per_class.get(c, 0) for c in CROPS)
axes[1].pie([weed_count, crop_count], labels=['Mauvaises herbes', 'Cultures'],
            colors=['#e74c3c', '#27ae60'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proportion Mauvaises herbes vs Cultures')
plt.tight_layout()
save_fig('01_distribution_classes.png')

# --- 12.2 Courbes d'entrainement ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
acc       = history.history['accuracy']    + history_ft.history['accuracy']
val_acc   = history.history['val_accuracy']+ history_ft.history['val_accuracy']
loss_vals     = history.history['loss']    + history_ft.history['loss']
val_loss_vals = history.history['val_loss']+ history_ft.history['val_loss']
ft_start = len(history.history['accuracy'])

axes[0].plot(acc,     label='Train Accuracy', color='#2196F3', linewidth=2)
axes[0].plot(val_acc, label='Val Accuracy',   color='#FF9800', linewidth=2)
axes[0].axvline(x=ft_start, color='green', linestyle='--', label='Fine-tuning')
axes[0].set_title('Accuracy par Epoque')
axes[0].set_xlabel('Epoque'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_vals,     label='Train Loss', color='#2196F3', linewidth=2)
axes[1].plot(val_loss_vals, label='Val Loss',   color='#FF9800', linewidth=2)
axes[1].axvline(x=ft_start, color='green', linestyle='--', label='Fine-tuning')
axes[1].set_title('Loss par Epoque')
axes[1].set_xlabel('Epoque'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_fig('02_training_curves.png')

# --- 12.3 Matrice de confusion ---
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Prediction'); ax.set_ylabel('Verite')
ax.set_title('Matrice de Confusion')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
save_fig('03_confusion_matrix.png')

# --- 12.4 Metriques par classe ---
fig, ax = plt.subplots(figsize=(14, 7))
class_metrics = {c: report[c] for c in CLASS_NAMES if c in report}
x_pos = np.arange(len(class_metrics))
width = 0.25
precisions = [class_metrics[c]['precision'] for c in class_metrics]
recalls    = [class_metrics[c]['recall']    for c in class_metrics]
f1s        = [class_metrics[c]['f1-score']  for c in class_metrics]
ax.bar(x_pos - width, precisions, width, label='Precision', color='#3498db')
ax.bar(x_pos,         recalls,    width, label='Recall',    color='#e74c3c')
ax.bar(x_pos + width, f1s,        width, label='F1-Score',  color='#2ecc71')
ax.set_xticks(x_pos)
ax.set_xticklabels(list(class_metrics.keys()), rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Precision, Recall et F1-Score par classe')
ax.legend(); ax.set_ylim(0, 1.1); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
save_fig('04_per_class_metrics.png')

# --- 12.5 Exemples de predictions ---
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
indices = random.sample(range(len(X_test)), 12)
for idx, ax in zip(indices, axes.flat):
    img_display = np.clip((X_test[idx] + 1.0) / 2.0, 0, 1)
    ax.imshow(img_display)
    true_label = CLASS_NAMES[y_test[idx]]
    pred_label = CLASS_NAMES[y_pred[idx]]
    correct = true_label == pred_label
    ax.set_title(f"Vrai: {true_label}\nPred: {pred_label}",
                 fontsize=9, color='green' if correct else 'red', fontweight='bold')
    ax.axis('off')
plt.suptitle('Exemples de Predictions (Vert=Correct, Rouge=Erreur)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('05_prediction_examples.png')

[OK] 01_distribution_classes.png
[OK] 02_training_curves.png
[OK] 03_confusion_matrix.png
[OK] 04_per_class_metrics.png
[OK] 05_prediction_examples.png


In [16]:
# ============================================================================
# RESUME FINAL
# ============================================================================
print("\n" + "=" * 60)
print("  RESUME DES RESULTATS")
print("=" * 60)
print(f"  Modele         : MobileNetV2 + Transfer Learning")
print(f"  Test Accuracy  : {test_acc*100:.2f}%")
print(f"  Test Loss      : {test_loss:.4f}")
print(f"  Macro F1-Score : {report['macro avg']['f1-score']:.4f}")
print(f"  Fichiers dans  : {OUTPUT_DIR}")
print(f"    - weed_detection_model.h5")
print(f"    - best_model.h5")
print(f"    - best_model_finetuned.h5")
print(f"    - 01_distribution_classes.png")
print(f"    - 02_training_curves.png")
print(f"    - 03_confusion_matrix.png")
print(f"    - 04_per_class_metrics.png")
print(f"    - 05_prediction_examples.png")
print("=" * 60)
print("  [OK] PROJET TERMINE AVEC SUCCES !")
print("=" * 60)


  RESUME DES RESULTATS
  Modele         : MobileNetV2 + Transfer Learning
  Test Accuracy  : 88.86%
  Test Loss      : 2.0278
  Macro F1-Score : 0.8119
  Fichiers dans  : /kaggle/working
    - weed_detection_model.h5
    - best_model.h5
    - best_model_finetuned.h5
    - 01_distribution_classes.png
    - 02_training_curves.png
    - 03_confusion_matrix.png
    - 04_per_class_metrics.png
    - 05_prediction_examples.png
  [OK] PROJET TERMINE AVEC SUCCES !
